# Chapter 2: Changing Independent Variables

Companion notebook to Section 2 of the study guide (Kundu, Cohen & Dowling 2016, Appendix B.2).
Write your own notes under each heading as you study.

Before committing: **Restart** the kernel, then **Run All**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.special import erf

from ch02_tools import (chi_transform, dune_bedload_flux, erf_step, frozen_pattern_rate,
                        scarp_max_slope, scarp_morphologic_age)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE = "#2a78d6"                           # one series
RAMP = ["#86b6ef", "#2a78d6", "#104281"]   # ordered series: early -> late
GRAY = "#52514e"                           # reference curves

## 2.1 The chain rule for new coordinates

$\dfrac{\partial}{\partial x} = \dfrac{\partial \xi}{\partial x}\dfrac{\partial}{\partial \xi} + \dfrac{\partial \psi}{\partial x}\dfrac{\partial}{\partial \psi} + \dfrac{\partial \zeta}{\partial x}\dfrac{\partial}{\partial \zeta} + \dfrac{\partial \tau}{\partial x}\dfrac{\partial}{\partial \tau}$ (B.2.1): you need the derivatives of the **new** variables with respect to the **old** ones.

Plane polar example: $\dfrac{\partial}{\partial x} = \cos\theta\,\dfrac{\partial}{\partial r} - \dfrac{\sin\theta}{r}\,\dfrac{\partial}{\partial \theta}$.

*My notes:*

In [ ]:
# Check d/dx = cos(theta) d/dr - sin(theta)/r d/dtheta on a test function
r, th = sp.symbols("r theta", positive=True)
x, y = sp.symbols("x y", real=True)
to_polar = {x: r * sp.cos(th), y: r * sp.sin(th)}

f_xy = x**3 * y + sp.exp(x) * sp.sin(y)   # any smooth function will do
f_rt = f_xy.subs(to_polar)                # the same field written in (r, theta)

direct = sp.diff(f_xy, x).subs(to_polar)
chain = sp.cos(th) * sp.diff(f_rt, r) - sp.sin(th) / r * sp.diff(f_rt, th)
print(sp.simplify(direct - chain))        # 0

### What is held constant?

$f = \sin(x - Ut)$ seen from a fixed point, and by an observer riding with the pattern ($\xi = x - Ut$, $\tau = t$, so $F = \sin\xi$).

*My notes:*

In [ ]:
x, t, U, xi, tau = sp.symbols("x t U xi tau", real=True)
f = sp.sin(x - U * t)    # f(x, t)
F = sp.sin(xi)           # the same field written as F(xi, tau)
print("fixed observer,  (df/dt)_x:    ", sp.diff(f, t))     # -U*cos(U*t - x) = -U cos(x - Ut)
print("riding observer, (dF/dtau)_xi: ", sp.diff(F, tau))   # 0

# B.2.2: (d/dt)_x = (d/dtau)_xi - U d/dxi
check = (sp.diff(F, tau) - U * sp.diff(F, xi)).subs(xi, x - U * t) - sp.diff(f, t)
print("B.2.2 check:", sp.simplify(check))                   # 0

## 2.2 Moving frames (Example B.2)

With $\xi = x - Ut$ and $\tau = t$: $\dfrac{\partial}{\partial x} = \dfrac{\partial}{\partial \xi}$ and $\dfrac{\partial}{\partial t} = \dfrac{\partial}{\partial \tau} - U\,\dfrac{\partial}{\partial \xi}$ (B.2.2).

A pattern that migrates without changing shape, $f = F(x - ct)$, obeys $\dfrac{\partial f}{\partial t} = -c\,\dfrac{\partial f}{\partial x}$.

*My notes:*

In [ ]:
# A train of asymmetric dunes migrating downstream (+x) at c = 1.5 m/h
H, stoss, lee = 0.8, 0.05, 0.6             # height [m]; stoss and lee slopes [-]
L_stoss, L_lee = H / stoss, H / lee        # lengths of the two faces [m]
wavelength = L_stoss + L_lee
c = 1.5                                    # celerity [m/h]


def dune_bed(xi):
    # bed elevation above the troughs [m] for a sawtooth dune train
    s = np.mod(xi, wavelength)
    return np.where(s < L_stoss, stoss * s, H - lee * (s - L_stoss))


x = np.linspace(0, 3 * wavelength + 6, 3000)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for t_h, color, style in zip([0, 2, 4], RAMP, ["-", "--", "-."]):
    z = dune_bed(x - c * t_h)
    axes[0].plot(x, z, style, color=color, label=f"t = {t_h} h")
    axes[1].plot(x - c * t_h, z, style, color=color)
axes[0].set(xlim=(0, 3 * wavelength), ylim=(-0.05, 1.15), xlabel="x (m), fixed frame",
            ylabel="bed elevation (m)", title="(a) fixed observer: the bed keeps changing")
axes[1].set(xlim=(0, 3 * wavelength), xlabel="xi = x - c t (m), moving frame",
            title="(b) riding with the dunes: one steady shape")
axes[0].text(0.5, 1.02, "flow and dunes move to the right", fontsize=8, color=GRAY)
axes[0].legend(loc="upper right", ncol=3, fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch02_moving_frame.png", dpi=200)
plt.show()

print(f"stoss side: dz/dt = {frozen_pattern_rate(c, stoss):+.3f} m/h (erosion)")
print(f"lee side:   dz/dt = {frozen_pattern_rate(c, -lee):+.3f} m/h (deposition)")

## 2.3 Similarity variables (Example B.3)

Try $f(x,t) = F(\eta)$ with $\eta = x\,t^{\alpha}$. Then $\dfrac{\partial f}{\partial x} = t^{\alpha} F'$, $\dfrac{\partial f}{\partial t} = \dfrac{\alpha}{t}\,\eta\,F'$ and $\dfrac{\partial^2 f}{\partial x^2} = t^{2\alpha} F''$.

Recipe: no intrinsic length or time scale, so combine $x$ and $t$; choose $\alpha$ so that $t$ cancels **and** the initial and boundary conditions collapse.

*My notes:*

In [ ]:
# Diffusion equation df/dt = kappa d2f/dx2 with the trial variable eta = x t^alpha
alpha = sp.symbols("alpha", real=True)
eta, t, kappa = sp.symbols("eta t kappa", positive=True)
Fp, Fpp = sp.symbols("Fp Fpp")          # dF/deta and d2F/deta2
lhs = alpha / t * eta * Fp              # df/dt
rhs = kappa * t**(2 * alpha) * Fpp      # kappa d2f/dx2
print(sp.powsimp(lhs / rhs))            # t appears only as t**(-2*alpha - 1)
print(sp.solve(sp.Eq(-2 * alpha - 1, 0), alpha))   # [-1/2], so eta ~ x / sqrt(t)

# With eta = x / sqrt(4 kappa t) the PDE becomes F'' + 2 eta F' = 0
F = sp.Function("F")
print(sp.dsolve(F(eta).diff(eta, 2) + 2 * eta * F(eta).diff(eta)))   # C1 + C2*erf(eta)

x = sp.symbols("x", real=True)
f = sp.erf(x / sp.sqrt(4 * kappa * t))
print(sp.simplify(sp.diff(f, t) - kappa * sp.diff(f, x, 2)))         # 0

In [ ]:
# Similarity collapse: a diffusing step (a degrading scarp with a = 2 m, no regional slope)
a = 2.0
x = np.linspace(-20, 20, 400)
eta_line = np.linspace(-2.5, 2.5, 200)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[1].plot(eta_line, a * erf(eta_line), color=GRAY, lw=1, label="a erf(eta)")
for i, (kt, color, style, marker) in enumerate(
        zip([2, 10, 40], RAMP, ["-", "--", "-."], ["o", "s", "^"])):
    axes[0].plot(x, erf_step(x, kt, a), style, color=color, label=f"kappa t = {kt} m^2")
    x_pts = (np.linspace(-2.2, 2.2, 12) + 0.13 * i) * np.sqrt(4 * kt)   # samples at each time
    axes[1].plot(x_pts / np.sqrt(4 * kt), erf_step(x_pts, kt, a), marker, color=color, ms=5,
                 ls="none", label=f"kappa t = {kt} m^2")
    print(f"kappa t = {kt:>2} m^2: diffusion length sqrt(4 kappa t) = {np.sqrt(4 * kt):5.2f} m")
axes[0].set(xlabel="x (m)", ylabel="z (m)", title="(a) profiles widen with time")
axes[1].set(xlim=(-2.5, 2.5), xlabel="eta = x / sqrt(4 kappa t)", ylabel="z (m)",
            title="(b) the same data versus eta")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch02_similarity_collapse.png", dpi=200)
plt.show()

## 2.4 Research connection: scarps, river profiles and migrating dunes

### Morphologic dating of a fault scarp

$z = a\,\mathrm{erf}\!\left(\dfrac{x}{\sqrt{4\kappa t}}\right) + b x$, steepest slope $S_{\max} = \dfrac{a}{\sqrt{\pi\kappa t}} + b$, so $\kappa t = \dfrac{a^2}{\pi (S_{\max}-b)^2}$.

*My notes:*

In [ ]:
kt = scarp_morphologic_age(a=2.0, b=0.05, s_max=0.30)   # 4 m offset, regional slope 0.05
print(f"morphologic age kappa t = {kt:.1f} m^2")                     # 20.4
print(f"age if kappa = 1 m^2/kyr: about {kt / 1.0:.0f} kyr")         # 20
print(f"check: S_max = {scarp_max_slope(kt, a=2.0, b=0.05):.2f}")    # 0.30

### The chi transform: the Chapter 1 knickpoint in chi space

$\chi(x) = \displaystyle\int_0^x \left(\dfrac{A_0}{A(x')}\right)^{m/n} dx'$. For $n = 1$, every knickpoint moves through $\chi$ at the same constant speed $K A_0^m$.

*My notes:*

In [ ]:
K, ka, L, A0 = 2e-6, 1.0, 20e3, 1.0              # same river as Chapter 1; A0 = 1 m^2
x = L * (1 - np.logspace(0, -3, 4000))            # 0 to 0.999 L, denser near the divide [m]
chi = chi_transform(x, ka * (L - x)**2, A0=A0, m_over_n=0.5)

t_yr = np.linspace(0, 3e6, 300)
x_k = L * (1 - np.exp(-K * np.sqrt(ka) * t_yr))   # knickpoint position from Chapter 1 [m]
chi_k = np.interp(x_k, x, chi)                    # the same positions in chi space [m]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(t_yr / 1e6, x_k / 1e3, color=BLUE)
axes[0].set(xlabel="time (Myr)", ylabel="distance upstream (km)", title="(a) slows down in x")
axes[1].plot(t_yr / 1e6, chi_k, color=BLUE, label="numerical chi")
axes[1].plot(t_yr / 1e6, K * A0**0.5 * t_yr, ":", color=GRAY, lw=2, label="K A0^m t")
axes[1].set(xlabel="time (Myr)", ylabel="chi (m)", title="(b) constant speed in chi")
axes[1].legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch02_chi_knickpoint.png", dpi=200)
plt.show()
print(f"chi of the knickpoint after 1 Myr: {np.interp(1e6, t_yr, chi_k):.3f} m")   # 2.000

### Dune tracking

$q_s = (1-\lambda_p)\,c\,z_b + q_0$; averaged over one wavelength, $\bar q_s \approx \beta\,(1-\lambda_p)\,c\,H$, with $\beta \approx 1/2$ for triangular dunes.

*My notes:*

In [ ]:
q = dune_bedload_flux(H=0.8, c=1.5 / 3600, porosity=0.35)   # celerity converted to m/s
print(f"bedload flux: {q:.2e} m^2/s of grains = {2650 * q:.2f} kg/m/s")   # 1.08e-04, 0.29

## Exercises 2.1-2.7

*Work space: solve each exercise here, then check against Appendix C of the study guide.*